# Dado Colab Quickstart

Install `dado[server]`, launch the async server in the background on Colab GPU (`bfloat16` default on CUDA), and run a 20-step SFT loop using the lightweight Python SDK.

In [ ]:
!pip install -q "dado[server]"

In [ ]:
import os
import subprocess
import sys
import time

import httpx

os.environ["DADO_API_KEYS"] = "colab-secret-key"
os.environ["DADO_DEVICE"] = "cuda"
os.environ["DADO_HOST"] = "127.0.0.1"

server_proc = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn",
        "dado.server.app:create_app", "--factory",
        "--host", "127.0.0.1", "--port", "8000",
    ],
    env=os.environ.copy(),
)

with httpx.Client(base_url="http://127.0.0.1:8000", timeout=2.0) as probe:
    for _ in range(60):
        try:
            if probe.get("/v1/healthz").status_code == 200:
                print("Server ready!")
                break
        except httpx.RequestError:
            pass
        time.sleep(0.5)

In [ ]:
from transformers import AutoTokenizer

from dado.client import OptimParams, SampleParams, ServiceClient

MODEL = "Qwen/Qwen2.5-0.5B"
tok = AutoTokenizer.from_pretrained(MODEL)
text = "dado runs LoRA fine-tuning primitives cleanly."
ids = tok(text, return_tensors="pt")["input_ids"][0].tolist()
batch = {
    "input_ids": [ids[:-1], ids[:-1]],
    "target_ids": [ids[1:], ids[1:]],
    "weights": [[1.0] * (len(ids) - 1)] * 2,
}

with (
    ServiceClient("http://127.0.0.1:8000", "colab-secret-key") as client,
    client.create_training_client(
        MODEL, 8, 16.0, ["q_proj", "v_proj"], device="cuda", dtype="bfloat16"
    ) as tc,
):
    for step in range(20):
        loss = tc.forward_backward(batch).result().loss
        tc.optim_step(OptimParams(lr=5e-4)).result()
        print(f"step={step + 1:02d} loss={loss:.4f}")
    sample = tc.sample(["dado runs"], SampleParams(max_tokens=8, temperature=0.0)).result()
    print("Sample:", sample.completions[0])

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=10)